# 附录B：符号、公式与算法选择

In [1]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / 'lsq').is_dir():
    if root == root.parent:
        raise RuntimeError('Run inside the repository')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from lsq import (finite_difference, whiten, gauss_newton,
                 levenberg_marquardt, dogleg, recursive_update)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (6.4, 3.3), 'font.size': 10,
                     'axes.grid': True, 'savefig.dpi': 130})

## 先按遇到的困难找工具

不用先背完所有缩写。先说“我的数据和目标是什么”，再选第一件值得试的工具。

| 你遇到的情况 | 可以从哪里开始 | 要同时检查什么 |
|---|---|---|
| 未知数只作为已知零件的系数 | `np.linalg.lstsq`，内部常用 SVD | 是否有重复信息，实际残差多大 |
| 有些读数明显更不可靠 | 按噪声尺度加权，必要时白化 | 噪声估计依据，误差会不会一起变化 |
| 线追着每个点乱摆 | 降低复杂度，或明确正则化 | 新数据是否变好，惩罚是否合理 |
| 曲线需要反复调整，初值较合理 | 带回溯的 GN | 影响表是否正确，大步是否失准 |
| 局部建议步太激进、没有边界要求 | LM | 参数尺度、接受和拒绝记录 |
| 想明确限制这次能调多远 | 信赖域、狗腿 | 真实改善是否兑现局部预测 |
| 参数有上下限，或有异常点 | SciPy `least_squares(method='trf')` | 初值合法，损失和噪声尺度匹配 |
| 线性读数不断进来 | RLS | 对象是否变化，要不要遗忘 |
| 仪器量化、平台、实测噪声明显 | 先设计有效扰动，再做受限局部校正 | 改动是否真的有响应，目标是否能达到 |

这是一张入口地图，不是按表选完就保证成功。尤其“数据分不清参数”和“局部步算得不好”，是不同困难。

## 符号速查：先说人话，再看维度

| 符号 | 白话意思 | 名称与大小 |
|---|---|---|
| $x,h$ | 当前旋钮设置、本轮改动 | 参数与步，均为 $n$ 个数 |
| $y,r$ | 实际读数、预测减读数 | 观测与残差，均为 $m$ 个数 |
| $A$ | 每个设置怎样产生各次预测的表 | 线性设计矩阵，$m\times n$ |
| $J$ | 每个旋钮微调后各次残差怎样变 | 雅可比，$m\times n$ |
| $F,g,H$ | 总分、总分的变化率、变化率怎样变 | 目标、梯度、Hessian |
| $C$ | 读数各自抖多少、是否一起抖 | 噪声协方差，$m\times m$ |
| $\mu,\Delta$ | 步长费用、这次允许走的半径 | 阻尼、信赖域半径 |
| $\lambda,D$ | 对最终参数罚多重、罚什么 | 正则化强度与惩罚算子 |

第13章的 $\lambda$ 另指遗忘因子；同一个字母在不同上下文可能换工作，不能只看符号猜含义。

## 六句能让结果被复查的话

先说模型和单位；说明残差及导数怎样检查；指出哪些参数组合看得清；记录为什么停止；用没参与拟合的数据检验；区分概率近似与工程容差。逐项回答，比不断换算法更容易找到问题。

## 公式速查，第二遍阅读时再用

线性最优条件：$A^T(Ax-y)=0$。非线性梯度：$g=J^Tr$。GN 的局部步：$\min_h\|Jh+r\|^2$。LM 的局部步：$\min_h\|Jh+r\|^2+\mu\|h\|^2$。预测下降：$-g^Th-\tfrac12\|Jh\|^2$。参数缩放 $x=Sz$ 后：$J_z=J_xS$。

## 下面这段诊断，先看哪几项

先认数组形状，再看秩和奇异值，最后看有限差分与手写雅可比的差。前两项在问信息够不够，最后一项在查变化率有没有写错。打印正常并不能替代残差图。

In [2]:
def diagnose_matrix(A):
    s=np.linalg.svd(A,compute_uv=False)
    rank=np.linalg.matrix_rank(A)
    full_column_rank=rank==A.shape[1]
    condition=(s[0]/s[-1]) if full_column_rank else np.inf
    return {'shape':A.shape,'rank':rank,'full_column_rank':full_column_rank,
            'condition':condition}
print(diagnose_matrix(np.column_stack([np.ones(5),np.arange(5.)])))
print(diagnose_matrix(np.ones((5,2))))
assert not diagnose_matrix(np.ones((5,2)))['full_column_rank']

{'shape': (5, 2), 'rank': np.int64(2), 'full_column_rank': np.True_, 'condition': np.float64(4.738720018687268)}
{'shape': (5, 2), 'rank': np.int64(1), 'full_column_rank': np.False_, 'condition': inf}


## 检查题

矩阵的秩、条件数和导数核对都正常，是否说明模型适合数据？

提示：一个计算得很准确的直线，能不能仍然不适合加速运动？

解答：可以不适合。矩阵诊断说明这套局部计算和信息结构怎样，残差图才帮助检查模型遗漏。计算正确、假设合理、预测有效，是要分别确认的几件事。